# Measure average time and cost for cv parser processing one document

In [37]:
from azure.ai.contentunderstanding import ContentUnderstandingClient
from azure.ai.contentunderstanding.models import AnalysisResult
from azure.core.credentials import AzureKeyCredential
from azure.core.exceptions import AzureError
import os
import pprint

In [2]:
# AZURE_CONTENT_UNDERSTANDING_ENDPOINT - the endpoint to your Content Understanding resource.
endpoint = os.environ["CONTENTUNDERSTANDING_ENDPOINT"]
# CONTENT_UNDERSTANDING_KEY - your Content Understanding API key
key = os.environ["CONTENTUNDERSTANDING_KEY"]

In [3]:
# API_VERSION - the API version to use.
api_version = "2025-11-01"

In [5]:
# Set up Content Understanding client.
credential = AzureKeyCredential(key)
client = ContentUnderstandingClient(endpoint=endpoint, credential=credential, api_version=api_version)

In [9]:
# ANALYZER_ID - the ID of the analyzer to use.
analyzer_id = "cv_parser_test"

In [26]:
# local file localtion
path_to_sample_document = "./examples/職務経歴書(サンプル).pdf"

In [40]:
def parse_cv(analyzer_id: str, path_to_sample_document:str):
    with open(path_to_sample_document, "rb") as f:
        poller = client.begin_analyze_binary(
            analyzer_id=analyzer_id,
            binary_input=f
        )
    result: AnalysisResult = poller.result()
    return result

In [47]:
%%timeit -r 7 -n 1 
# Calulate average time cv parsing cost per document (2-page pdf)
result = parse_cv(analyzer_id, path_to_sample_document)

22.9 s ± 1.38 s per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [65]:
# According to Azure  Content Understanding in Foundry Tools pricing in https://azure.microsoft.com/en-us/pricing/details/content-understanding/#pricing
# and Azure OpenAI Service pricing in https://azure.microsoft.com/en-us/pricing/details/azure-openai/
price_dict = {
    # in our usecase, uploaded cv can be in pdf, docx, xlsx format and it require structural element detection from image-based documents.
    # We have to use standard meter here according to the instruction https://learn.microsoft.com/en-us/azure/ai-services/content-understanding/pricing-explainer#example-1-document-processing-for-rag-workflows. 
    'documentPagesStandard':  5 / 1000,
    'contextualizationTokens': 1 /1000000,
    'tokens': {'text-embedding-3-large': 0.000158/1000, 
               'gpt-5-input': 1.25 / 1000000,
               'gpt-5-output': 10 /1000000
}
}

In [68]:
# Calulate total cost in usd for running cv parsing per document (2-page pdf)
# Caluation is based on https://azure.microsoft.com/en-us/pricing/details/azure-openai/
total_cost = 0
for charge_item, item_count in poller.usage.items():
    if charge_item == "tokens":
        for sub_charge_item, sub_item_count in item_count.items():
            total_cost += price_dict[charge_item][sub_charge_item] * sub_item_count
        
    else:
        total_cost += price_dict[charge_item]*item_count
print(f"Total cost (usd) per document(2-page pdf): {total_cost}")

Total cost (usd) per document(2-page pdf): 0.036237992


In [39]:
# Parsied result
content = result.contents[0]
pprint.pprint(content.fields, sort_dicts=False)

{'学歴': {'type': 'array'},
 '職歴': {'type': 'array', 'valueArray': [{'type': 'object', 'valueObject': {'会社名': {'type': 'string', 'valueString': '株式会社', 'spans': [{'offset': 273, 'length': 4}], 'confidence': 0.497, 'source': 'D(1,2.7439,3.9964,3.2805,4.0001,3.2805,4.1387,2.7439,4.1424)'}, '在籍期間': {'type': 'object', 'valueObject': {'開始日': {'type': 'date', 'valueDate': '2007-04-01', 'spans': [{'offset': 257, 'length': 7}], 'confidence': 0.51, 'source': 'D(1,0.9356,3.9983,1.6244,3.9998,1.6241,4.1403,0.9353,4.1389)'}, '終了日': {'type': 'date', 'valueDate': '2018-01-01', 'spans': [{'offset': 265, 'length': 7}], 'confidence': 0.469, 'source': 'D(1,1.7755,3.9992,2.4878,4.0003,2.4876,4.1421,1.7753,4.1410)'}}}, '事業内容': {'type': 'string', 'valueString': 'インターネット広告事業', 'spans': [{'offset': 293, 'length': 11}], 'confidence': 0.651, 'source': 'D(1,1.5981,4.4793,3.0315,4.4889,3.0304,4.6523,1.5970,4.6427)'}, '業種': {'type': 'string', 'confidence': 0.863}, '職種': {'type': 'string', 'valueString': '経理', 'span

### Measure time it take to process document in word format

In [ ]:
path_to_sample_document = "./examples/docx/職務経歴書(サンプル).docx"

In [69]:
%%timeit -r 7 -n 1 
# Calulate average time cv parsing cost per document (2-page docx)
result = parse_cv(analyzer_id, path_to_sample_document)

23 s ± 1.27 s per loop (mean ± std. dev. of 7 runs, 1 loop each)


### Measure time it take to process document in excel format

In [70]:
path_to_sample_document = "./examples/xlsx/職務経歴書(サンプル).xlsx"

In [71]:
%%timeit -r 7 -n 1 
# Calulate average time cv parsing cost per document (2-page xlsx)
result = parse_cv(analyzer_id, path_to_sample_document)

26.5 s ± 3.18 s per loop (mean ± std. dev. of 7 runs, 1 loop each)
